In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
from engression.loss_func import energy_loss, energy_loss_two_sample
import scipy.stats as stats
from graphlib import TopologicalSorter, _NodeInfo

In [2]:
df = pd.read_csv('PoliticalDemocracy.csv')
var_names = df.columns.values.tolist()
dataset = torch.tensor(df.to_numpy()).to(dtype = torch.float32)
batch_size = 25
dataloader = DataLoader(TensorDataset(dataset), batch_size = batch_size, shuffle = True)
print(var_names)

['y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'x1', 'x2', 'x3']


In [3]:
class rv_sem2(_NodeInfo):
    def __init__(self, formula: str):

        if "<-" in formula:
            lhs, rhs = formula.replace(" ", "").split("<-")
            super().__init__(lhs) # lhs is the variable name
            self.predecessors = rhs.split("+") # rhs is the parents of the node
            self.dist = None
            self.is_random = False
        elif "~" in formula:
            lhs, rhs = formula.replace(" ", "").split("~")
            super().__init__(lhs) # lhs is the variable name
            self.predecessors = []
            self.is_random = True
            self.dist = eval("stats."+rhs+"()") # rhs is the distribution (must be a class in scipy.stats)
        else:
            raise ValueError("Formula does not contain <- or ~.")

        self.npredecessors = len(self.predecessors) # NOTE: this will be changed when obtaining topological order
        self.formula = formula

    def __repr__(self):
        return self.formula

In [4]:
class sem2(TopologicalSorter, nn.Module):
    def __init__(self, model_formula = None):

        nn.Module.__init__(self)
        TopologicalSorter.__init__(self, graph = None)

        formula_list = model_formula.replace(" ", "").split(";")

        # Initialize node info by formula
        var = [rv_sem2(formula) for formula in formula_list]
        self._node2info = {v.node:v for v in var}                
        
        for node, v in self._node2info.items():                
            for p in v.predecessors:
                self._node2info[p].successors.append(node)

        self.layer = nn.ModuleDict()
        for node, v in self._node2info.items():
            if v.is_random is True:
                self.layer[node] = None
            else:
                self.layer[node] = nn.Linear(len(v.predecessors), 1)

        self._topological_order = list(super(sem2, self).static_order())
        self._node2index = {node: index for index, node in enumerate(self._topological_order)}
    
    def _get_nodeinfo(self, node: str):
        """Override original private method, raise an error if node does not exist
        """
        return self._node2info[node]

    def add(self, node, *predecessors):
        """Override original methods, don't do anything
        """
        return None

    def static_order(self):
        """Override static_order to return cached topological order"""
        return iter(self._topological_order)

    def forward(self, batch_size = 1):
        n_nodes = len(self._node2info)
        x = torch.zeros(batch_size, n_nodes)
        for node in self._topological_order:
            v = self._node2info[node]
            index = self._node2index[node]
            if v.is_random is True:
                 x[:, index] = torch.tensor(v.dist.rvs(batch_size))
            else:
                pred_indices = [self._node2index[p] for p in v.predecessors]
                x[:, index] = self.layer[node](x[:, pred_indices]).squeeze(-1)
        return x

In [5]:
model_formula = "X <- A + B + C; B <- A + C; A ~ uniform; C ~ norm"
model = sem2(model_formula)
var = model._node2info
[var, 
 var["A"].dist.rvs(10), var["C"].dist.rvs(10), 
 var["A"].successors, var["B"].predecessors, model._topological_order]

[{'X': X<-A+B+C, 'B': B<-A+C, 'A': A~uniform, 'C': C~norm},
 array([0.08767618, 0.91832697, 0.94635848, 0.95857007, 0.56019363,
        0.53417788, 0.47591196, 0.93213024, 0.13414278, 0.03878628]),
 array([ 0.52825196, -0.02653201, -0.96517329,  1.54042515, -0.35716726,
        -0.2820985 , -0.28114596, -0.07464935, -1.7611612 , -0.34855243]),
 ['X', 'B'],
 ['A', 'C'],
 ['A', 'C', 'B', 'X']]

In [6]:
model(8)

tensor([[ 0.5368,  1.4603, -0.8225,  0.2825],
        [ 0.0046,  0.4046, -0.5980,  0.2202],
        [ 0.9223, -0.0109, -0.3906,  0.5639],
        [ 0.8129,  0.5856, -0.5608,  0.4651],
        [ 0.7328,  0.5272, -0.5537,  0.4452],
        [ 0.2330,  0.9868, -0.7286,  0.2333],
        [ 0.5792, -0.7459, -0.2314,  0.5295],
        [ 0.1640, -1.3766, -0.1075,  0.4605]], grad_fn=<CopySlices>)

In [7]:
# Linear factor model
model_formula = "x1 <- ind60 + ex1; x2 <- ind60 + ex2; x3 <- ind60 + ex3; ind60 ~ norm; ex1 ~ norm; ex2 ~ norm; ex3 ~ norm"
model = sem2(model_formula)
observed_var = ["x1", "x2", "x3"]
x = dataset[:, [var_names.index(var) for var in observed_var]]
observed_index = [list(model.static_order()).index(var) for var in observed_var]

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model.train()
for i in range(1000):
    optimizer.zero_grad()
    sample1 = model(x.size(0))[:, observed_index]
    sample2 = model(x.size(0))[:, observed_index]
    loss, s1, s2 = energy_loss_two_sample(x, sample1, sample2)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 100 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')

Iter 1 | loss: 7.4191, s1: 8.1518, s2: 1.4654
Iter 100 | loss: 4.6738, s1: 6.7960, s2: 4.2444
Iter 200 | loss: 3.4703, s1: 6.2965, s2: 5.6524
Iter 300 | loss: 2.9624, s1: 5.8193, s2: 5.7138
Iter 400 | loss: 2.7197, s1: 4.9561, s2: 4.4727
Iter 500 | loss: 1.7697, s1: 4.2894, s2: 5.0395
Iter 600 | loss: 1.3094, s1: 3.4035, s2: 4.1881
Iter 700 | loss: 1.2844, s1: 3.1741, s2: 3.7795
Iter 800 | loss: 1.4157, s1: 2.8540, s2: 2.8766
Iter 900 | loss: 1.3155, s1: 2.6324, s2: 2.6339
Iter 1000 | loss: 1.5262, s1: 2.7744, s2: 2.4964


In [9]:
beta = [model.layer[node].weight[0][0].squeeze().detach().numpy().tolist() for node in observed_var]
[beta[i] / beta[0] for i in range(3)]

[1.0, 2.1129665515207683, 1.9605148078024355]

In [10]:
# # Linear factor model (same as lavaan package example)
# formula_epsx = "ex1 ~ norm; ex2 ~ norm; ex3 ~ norm;" 
# formula_epsy = "ey1 ~ norm; ey2 ~ norm; ey3 ~ norm; ey4 ~ norm; ey5 ~ norm; ey6 ~ norm; ey7 ~ norm; ey8 ~ norm;"
# formula_epsdem = "edem60 ~ norm; edem65 ~ norm;"
# formula_epscov = "e15 ~ norm; e24 ~ norm; e26 ~ norm; e37 ~ norm; e48 ~ norm; e68 ~ norm;"
# formula_end = "ind60 ~ norm; dem60 <- ind60 + edem60; dem65 <- ind60 + dem60 + edem65;"
# formula_x = "x1 <- ind60 + ex1; x2 <- ind60 + ex2; x3 <- ind60 + ex3;"
# formula_y1 = "y1 <- dem60 + ey1 + e15; y2 <- dem60 + ey2 + e24 + e26; y3 <- dem60 + ey3 + e37; y4 <- dem60 + ey4 + e24 + e48;"
# formula_y2 = "y5 <- dem65 + ey5 + e15; y6 <- dem65 + ey6 + e26; y7 <- dem65 + ey7 + e37; y8 <- dem65 + ey8 + e48 + e68"
# formula = formula_epsx + formula_epsy + formula_epsdem + formula_epscov + formula_end + formula_x + formula_y1 + formula_y2

In [11]:
# Linear factor model (without residual correlations)
formula_epsx = "ex1 ~ norm; ex2 ~ norm; ex3 ~ norm;" 
formula_epsy = "ey1 ~ norm; ey2 ~ norm; ey3 ~ norm; ey4 ~ norm; ey5 ~ norm; ey6 ~ norm; ey7 ~ norm; ey8 ~ norm;"
formula_epsdem = "edem60 ~ norm; edem65 ~ norm;"
formula_end = "ind60 ~ norm; dem60 <- ind60 + edem60; dem65 <- ind60 + dem60 + edem65;"
formula_x = "x1 <- ind60 + ex1; x2 <- ind60 + ex2; x3 <- ind60 + ex3;"
formula_y1 = "y1 <- dem60 + ey1; y2 <- dem60 + ey2; y3 <- dem60 + ey3; y4 <- dem60 + ey4;"
formula_y2 = "y5 <- dem65 + ey5; y6 <- dem65 + ey6; y7 <- dem65 + ey7; y8 <- dem65 + ey8"
formula = formula_epsx + formula_epsy + formula_epsdem + formula_end + formula_x + formula_y1 + formula_y2

In [12]:
observed_var = var_names
x = dataset[:, [var_names.index(var) for var in observed_var]]
model = sem2(formula)
observed_index = [list(model.static_order()).index(var) for var in observed_var]

In [13]:
model = sem2(formula)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model.train()
for i in range(5000):
    optimizer.zero_grad()
#    sample1 = model(x.size(0))[:, observed_index]
#    sample2 = model(x.size(0))[:, observed_index]
#    loss, s1, s2 = energy_loss_two_sample(x, sample1, sample2)
    sample = model(x.size(0) * 100)[:, observed_index]
    loss, s1, s2 = energy_loss(x, sample)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 500 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')
        ind60_scale = model.layer["x1"].weight[0][0]
        dem60_scale = model.layer["y1"].weight[0][0]
        dem65_scale = model.layer["y5"].weight[0][0]
        print([model.layer["dem60"].weight[0][0] / ind60_scale * dem60_scale,
               model.layer["dem65"].weight[0][0] / ind60_scale * dem65_scale,
               model.layer["dem65"].weight[0][1] / dem60_scale * dem65_scale])

Iter 1 | loss: 16.7602, s1: 17.9525, s2: 2.3845
[tensor(2.4994, grad_fn=<MulBackward0>), tensor(-0.7530, grad_fn=<MulBackward0>), tensor(0.1887, grad_fn=<MulBackward0>)]
Iter 500 | loss: 6.3125, s1: 12.8123, s2: 12.9996
[tensor(7.7900, grad_fn=<MulBackward0>), tensor(-0.3145, grad_fn=<MulBackward0>), tensor(1.0205, grad_fn=<MulBackward0>)]
Iter 1000 | loss: 6.2424, s1: 12.5590, s2: 12.6332
[tensor(2.7009, grad_fn=<MulBackward0>), tensor(0.6714, grad_fn=<MulBackward0>), tensor(0.9580, grad_fn=<MulBackward0>)]
Iter 1500 | loss: 6.0888, s1: 12.4215, s2: 12.6654
[tensor(2.3108, grad_fn=<MulBackward0>), tensor(0.6855, grad_fn=<MulBackward0>), tensor(0.8882, grad_fn=<MulBackward0>)]
Iter 2000 | loss: 6.1961, s1: 12.5947, s2: 12.7972
[tensor(2.7339, grad_fn=<MulBackward0>), tensor(0.9043, grad_fn=<MulBackward0>), tensor(0.8263, grad_fn=<MulBackward0>)]
Iter 2500 | loss: 6.1174, s1: 12.4870, s2: 12.7393
[tensor(2.0935, grad_fn=<MulBackward0>), tensor(0.8398, grad_fn=<MulBackward0>), tensor(0.8

In [14]:
# change loss function to OLS (between sample and model covariance matrices)
sample_cov = torch.cov(torch.t(x))
model = sem2(formula)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-2)
model.train()
for i in range(5000):
    optimizer.zero_grad()
    model_sample = model(x.size(0) * 10)[:, observed_index]
    model_cov = torch.cov(torch.t(model_sample))
    loss = torch.norm(sample_cov - model_cov)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 500 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}')
        ind60_scale = model.layer["x1"].weight[0][0]
        dem60_scale = model.layer["y1"].weight[0][0]
        dem65_scale = model.layer["y5"].weight[0][0]
        print([model.layer["dem60"].weight[0][0] / ind60_scale * dem60_scale,
               model.layer["dem65"].weight[0][0] / ind60_scale * dem65_scale,
               model.layer["dem65"].weight[0][1] / dem60_scale * dem65_scale])

Iter 1 | loss: 59.1245
[tensor(-0.3186, grad_fn=<MulBackward0>), tensor(0.2827, grad_fn=<MulBackward0>), tensor(-0.1606, grad_fn=<MulBackward0>)]
Iter 500 | loss: 10.7053
[tensor(2.7397, grad_fn=<MulBackward0>), tensor(1.4003, grad_fn=<MulBackward0>), tensor(0.6966, grad_fn=<MulBackward0>)]
Iter 1000 | loss: 7.9102
[tensor(2.3927, grad_fn=<MulBackward0>), tensor(1.0367, grad_fn=<MulBackward0>), tensor(0.7093, grad_fn=<MulBackward0>)]
Iter 1500 | loss: 14.4583
[tensor(1.8302, grad_fn=<MulBackward0>), tensor(0.7052, grad_fn=<MulBackward0>), tensor(0.7675, grad_fn=<MulBackward0>)]
Iter 2000 | loss: 5.5441
[tensor(1.4160, grad_fn=<MulBackward0>), tensor(0.4570, grad_fn=<MulBackward0>), tensor(0.7971, grad_fn=<MulBackward0>)]
Iter 2500 | loss: 5.5564
[tensor(1.2787, grad_fn=<MulBackward0>), tensor(0.4401, grad_fn=<MulBackward0>), tensor(0.8647, grad_fn=<MulBackward0>)]
Iter 3000 | loss: 8.0317
[tensor(1.2450, grad_fn=<MulBackward0>), tensor(0.4342, grad_fn=<MulBackward0>), tensor(0.8906, gr

In [15]:
# change loss function to negative log likelihood
sample_cov = torch.cov(torch.t(x))
model = sem2(formula)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-2)
model.train()
for i in range(5000):
    optimizer.zero_grad()
    model_sample = model(x.size(0) * 10)[:, observed_index]
    model_cov = torch.cov(torch.t(model_sample))
    loss = torch.log(torch.det(model_cov)) + torch.trace(sample_cov @ torch.inverse(model_cov))
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 500 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}')
        ind60_scale = model.layer["x1"].weight[0][0]
        dem60_scale = model.layer["y1"].weight[0][0]
        dem65_scale = model.layer["y5"].weight[0][0]
        print([model.layer["dem60"].weight[0][0] / ind60_scale * dem60_scale,
               model.layer["dem65"].weight[0][0] / ind60_scale * dem65_scale,
               model.layer["dem65"].weight[0][1] / dem60_scale * dem65_scale])

Iter 1 | loss: 2205.7046
[tensor(-0.0871, grad_fn=<MulBackward0>), tensor(0.5253, grad_fn=<MulBackward0>), tensor(-0.2527, grad_fn=<MulBackward0>)]
Iter 500 | loss: 26.0911
[tensor(1.8989, grad_fn=<MulBackward0>), tensor(2.2705, grad_fn=<MulBackward0>), tensor(0.4218, grad_fn=<MulBackward0>)]
Iter 1000 | loss: 24.5254
[tensor(1.8203, grad_fn=<MulBackward0>), tensor(1.7840, grad_fn=<MulBackward0>), tensor(0.4678, grad_fn=<MulBackward0>)]
Iter 1500 | loss: 22.7759
[tensor(1.2782, grad_fn=<MulBackward0>), tensor(0.8190, grad_fn=<MulBackward0>), tensor(0.6261, grad_fn=<MulBackward0>)]
Iter 2000 | loss: 22.4881
[tensor(1.4115, grad_fn=<MulBackward0>), tensor(0.4701, grad_fn=<MulBackward0>), tensor(0.6659, grad_fn=<MulBackward0>)]
Iter 2500 | loss: 22.1421
[tensor(1.4763, grad_fn=<MulBackward0>), tensor(0.3633, grad_fn=<MulBackward0>), tensor(0.6866, grad_fn=<MulBackward0>)]
Iter 3000 | loss: 22.2881
[tensor(1.4394, grad_fn=<MulBackward0>), tensor(0.3513, grad_fn=<MulBackward0>), tensor(0.69

In [16]:
x = dataset[:, [var_names.index(var) for var in observed_var]]
sd = torch.std(x, dim = 0)
print(sd)
x_normalized = torch.zeros_like(x)
for i in range(x.size(1)):
    x_normalized[:,i] = x[:,i] / sd[i]

tensor([2.6227, 3.9471, 3.2809, 3.3495, 2.6126, 3.3727, 3.2862, 3.2456, 0.7329,
        1.5107, 1.4057])


In [ ]:
# Try normalizing the columns before training.
model = sem2(formula)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model.train()
for i in range(5000):
    optimizer.zero_grad()
#    sample1 = model(x.size(0))[:, observed_index]
#    sample2 = model(x.size(0))[:, observed_index]
#    loss, s1, s2 = energy_loss_two_sample(x, sample1, sample2)
    sample = model(x.size(0) * 100)[:, observed_index]
    loss, s1, s2 = energy_loss(x_normalized, sample)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 500 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')
        ind60_scale = model.layer["x1"].weight[0][0] * sd[var_names.index("x1")]
        dem60_scale = model.layer["y1"].weight[0][0] * sd[var_names.index("y1")]
        dem65_scale = model.layer["y5"].weight[0][0] * sd[var_names.index("y5")]
        print([model.layer["dem60"].weight[0][0] / ind60_scale * dem60_scale,
               model.layer["dem65"].weight[0][0] / ind60_scale * dem65_scale,
               model.layer["dem65"].weight[0][1] / dem60_scale * dem65_scale])

Iter 1 | loss: 9.0291, s1: 9.9029, s2: 1.7475
[tensor(-1.2423, grad_fn=<MulBackward0>), tensor(-0.6076, grad_fn=<MulBackward0>), tensor(-0.0687, grad_fn=<MulBackward0>)]
Iter 500 | loss: 2.9564, s1: 6.2130, s2: 6.5134
[tensor(0.4315, grad_fn=<MulBackward0>), tensor(0.1975, grad_fn=<MulBackward0>), tensor(0.2254, grad_fn=<MulBackward0>)]
Iter 1000 | loss: 2.2289, s1: 4.5077, s2: 4.5576
[tensor(1.9881, grad_fn=<MulBackward0>), tensor(0.8706, grad_fn=<MulBackward0>), tensor(0.7426, grad_fn=<MulBackward0>)]
